# Data Preparation and EDA for RecoMart Recommendation Pipeline

This notebook prepares the validated data for modeling by:
- Further cleaning if needed
- Encoding categorical columns
- Normalizing numeric columns
- Running exploratory data analysis with plots

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# Plot configuration
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Resolve project paths reliably whether the notebook is launched from:
# 1) the project root, or
# 2) the notebooks directory.
cwd = Path.cwd().resolve()
if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

VALIDATED_DIR = PROJECT_ROOT / "validated"
PROCESSED_DIR = PROJECT_ROOT / "processed"
PLOTS_DIR = PROJECT_ROOT / "reports" / "eda_summary_plots"

clickstream_path = VALIDATED_DIR / "clickstream_validated.csv"
products_path = VALIDATED_DIR / "products_validated.csv"

for path in (clickstream_path, products_path):
    if not path.exists():
        raise FileNotFoundError(
            f"Required input file not found: {path}\n"
            "Run the validation stage first or verify the project folder structure."
        )

clickstream_df = pd.read_csv(clickstream_path)
products_df = pd.read_csv(products_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Clickstream shape: {clickstream_df.shape}")
print(f"Products shape: {products_df.shape}")

required_clickstream = {"user_id", "product_id", "event_time", "event_type"}
required_products = {"product_id", "category_code", "brand", "price"}

missing_clickstream = required_clickstream - set(clickstream_df.columns)
missing_products = required_products - set(products_df.columns)

if missing_clickstream:
    raise ValueError(
        f"Clickstream file is missing required columns: {sorted(missing_clickstream)}"
    )
if missing_products:
    raise ValueError(
        f"Products file is missing required columns: {sorted(missing_products)}"
    )

# Avoid duplicate product rows multiplying clickstream records during the merge.
products_for_merge = (
    products_df[["product_id", "category_code", "brand", "price"]]
    .drop_duplicates(subset=["product_id"], keep="last")
    .copy()
)

# Keep product-master attributes as the source of truth.
overlapping_product_columns = [
    column
    for column in ("category_code", "brand", "price")
    if column in clickstream_df.columns
]
if overlapping_product_columns:
    clickstream_df = clickstream_df.drop(columns=overlapping_product_columns)

merged_df = clickstream_df.merge(
    products_for_merge,
    on="product_id",
    how="left",
    validate="many_to_one",
)

unmatched_products = merged_df["price"].isna().sum()
if unmatched_products:
    print(
        f"Warning: {unmatched_products:,} clickstream rows did not match a product record."
    )

print(f"\nMerged shape: {merged_df.shape}")
print("Data loaded and merged successfully!")


## Encode Categorical Columns

We use different encoding strategies based on the nature of each column:

- **event_type (ordinal encoding)**: The order matters - view < cart < purchase represents increasing user intent, so we assign weights (view=1, cart=2, purchase=3)
- **category_code (frequency encoding)**: Has many unique values, so one-hot would create too many columns. Frequency encoding replaces each category with its occurrence count
- **brand (frequency encoding)**: Same as category_code - too many unique brands for one-hot encoding
- **day_of_week (one-hot encoding)**: Only 7 possible values, so one-hot is manageable and preserves the categorical nature without implying order

In [ ]:
# Parse event_time safely.
merged_df["event_time"] = pd.to_datetime(
    merged_df["event_time"], errors="coerce", utc=True
)

invalid_event_times = merged_df["event_time"].isna().sum()
if invalid_event_times:
    print(f"Warning: dropping {invalid_event_times:,} rows with invalid event_time.")
    merged_df = merged_df.dropna(subset=["event_time"]).copy()

# Fill missing categorical values before encoding.
merged_df["event_type"] = (
    merged_df["event_type"].astype("string").str.strip().str.lower()
)
merged_df["category_code"] = (
    merged_df["category_code"].astype("string").fillna("unknown")
)
merged_df["brand"] = merged_df["brand"].astype("string").fillna("unknown")

merged_df["day_of_week"] = merged_df["event_time"].dt.day_name()

# Event-weight encoding represents increasing purchase intent.
event_type_mapping = {
    "page_view": 1,
    "click": 1,
    "view": 1,
    "search": 1,
    "login": 1,
    "logout": 1,
    "signup": 1,
    "add_to_cart": 2,
    "cart": 2,
    "purchase": 3,
}
merged_df["event_weight"] = (
    merged_df["event_type"].map(event_type_mapping).fillna(1).astype("int8")
)

# Frequency encoding for high-cardinality categorical attributes.
category_freq = merged_df["category_code"].value_counts(normalize=True)
brand_freq = merged_df["brand"].value_counts(normalize=True)

merged_df["category_code_encoded"] = (
    merged_df["category_code"].map(category_freq).fillna(0.0)
)
merged_df["brand_encoded"] = (
    merged_df["brand"].map(brand_freq).fillna(0.0)
)

# One-hot encoding for day of week. dtype=int keeps CSV output compact and readable.
day_of_week_dummies = pd.get_dummies(
    merged_df["day_of_week"],
    prefix="day_of_week",
    dtype="int8",
)
merged_df = pd.concat([merged_df, day_of_week_dummies], axis=1)

print("Encoding completed.")
print(f"DataFrame shape after encoding: {merged_df.shape}")


## Normalize Numeric Columns

We normalize numeric columns like price and hour_of_day to a 0-1 range using MinMaxScaler. This is important because raw price values (e.g., $10-$1000) are on a completely different scale than other features like event counts or encoded categories. Without normalization, price would dominate distance-based similarity calculations and model training, making other features less impactful.

In [ ]:
# Convert price to numeric and handle invalid/missing values.
merged_df["price"] = pd.to_numeric(merged_df["price"], errors="coerce")

if merged_df["price"].notna().sum() == 0:
    raise ValueError("The price column contains no valid numeric values.")

price_median = merged_df["price"].median()
missing_price_count = merged_df["price"].isna().sum()

if missing_price_count:
    print(
        f"Warning: filling {missing_price_count:,} missing/invalid prices "
        f"with median price {price_median:.2f}."
    )
    merged_df["price"] = merged_df["price"].fillna(price_median)

merged_df["hour_of_day"] = merged_df["event_time"].dt.hour.astype("int8")

price_scaler = MinMaxScaler()
merged_df["price_normalized"] = price_scaler.fit_transform(
    merged_df[["price"]]
).ravel()

# Hour has a known fixed range (0-23); direct scaling is stable even if the sample
# does not contain every hour.
merged_df["hour_normalized"] = merged_df["hour_of_day"] / 23.0

print("Price normalization:")
print(
    f"  Original price - Min: {merged_df['price'].min():.2f}, "
    f"Max: {merged_df['price'].max():.2f}, "
    f"Mean: {merged_df['price'].mean():.2f}"
)
print(
    f"  Normalized price - Min: {merged_df['price_normalized'].min():.4f}, "
    f"Max: {merged_df['price_normalized'].max():.4f}, "
    f"Mean: {merged_df['price_normalized'].mean():.4f}"
)

print("\nHour-of-day normalization:")
print(
    f"  Original hour - Min: {merged_df['hour_of_day'].min()}, "
    f"Max: {merged_df['hour_of_day'].max()}, "
    f"Mean: {merged_df['hour_of_day'].mean():.2f}"
)
print(
    f"  Normalized hour - Min: {merged_df['hour_normalized'].min():.4f}, "
    f"Max: {merged_df['hour_normalized'].max():.4f}, "
    f"Mean: {merged_df['hour_normalized'].mean():.4f}"
)

print(f"\nDataFrame shape after normalization: {merged_df.shape}")


## Exploratory Data Analysis

Now we'll create some plots to understand the data distribution and patterns. Each plot is saved as a PNG file in the reports/eda_summary_plots/ directory for the final deliverable.

In [ ]:
# Create the output directory once and reuse it for every plot.
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOTS_DIR}")


# 1. Histogram of price distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=merged_df, x="price", bins=50, kde=True)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plot_path = PLOTS_DIR / "price_distribution.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {plot_path}")


In [ ]:
# 2. Event counts by hour of day
hourly_counts = (
    merged_df["hour_of_day"]
    .value_counts()
    .reindex(range(24), fill_value=0)
    .sort_index()
)

plt.figure(figsize=(10, 6))
sns.barplot(x=hourly_counts.index, y=hourly_counts.values)
plt.title("Events per Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Events")
plt.xticks(range(24))
plt.tight_layout()
plot_path = PLOTS_DIR / "hourly_activity.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {plot_path}")


In [ ]:
# 3. Bar chart of event-type counts
event_counts = merged_df["event_type"].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=event_counts.index, y=event_counts.values)
plt.title("Event Type Counts")
plt.xlabel("Event Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plot_path = PLOTS_DIR / "event_type_counts.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {plot_path}")


In [ ]:
# 4. Top 10 categories by event count
top_categories = merged_df["category_code"].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_categories.index.astype(str), y=top_categories.values)
plt.title("Top 10 Categories by Event Count")
plt.xlabel("Category Code")
plt.ylabel("Number of Events")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plot_path = PLOTS_DIR / "top_categories.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {plot_path}")


In [ ]:
# 5. Correlation heatmap
# Avoid including both price and price_normalized because they are mathematically
# equivalent and would add a redundant perfect correlation.
numeric_cols = [
    "price",
    "event_weight",
    "hour_of_day",
    "category_code_encoded",
    "brand_encoded",
]
correlation_matrix = merged_df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plot_path = PLOTS_DIR / "correlation_heatmap.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {plot_path}")


## Save Final Features for Feature Engineering

Now we'll select the columns needed for the next stage and save the processed data to a CSV file. This includes the original identifiers, timestamps, categorical columns (both original and encoded), numeric columns (both original and normalized), and the one-hot encoded day of week columns.

In [ ]:
# Select columns needed for downstream feature engineering.
base_feature_columns = [
    "user_id",
    "product_id",
    "event_time",
    "event_type",
    "event_weight",
    "category_code",
    "category_code_encoded",
    "brand",
    "brand_encoded",
    "price",
    "price_normalized",
    "hour_of_day",
    "hour_normalized",
]

# user_session is optional because some clickstream datasets do not contain it.
if "user_session" in merged_df.columns:
    base_feature_columns.append("user_session")
else:
    print("Note: user_session is not present and will not be included.")

day_columns = sorted(
    column for column in merged_df.columns if column.startswith("day_of_week_")
)
feature_columns = base_feature_columns + day_columns

missing_final_columns = [
    column for column in feature_columns if column not in merged_df.columns
]
if missing_final_columns:
    raise ValueError(
        f"Cannot create final feature dataset. Missing columns: {missing_final_columns}"
    )

features_final = merged_df[feature_columns].copy()

# Remove exact duplicate interaction rows in the prepared output.
before_dedup = len(features_final)
features_final = features_final.drop_duplicates().reset_index(drop=True)
removed_duplicates = before_dedup - len(features_final)

print(f"Final features shape: {features_final.shape}")
print(f"Columns included: {len(feature_columns)}")
print(f"Duplicate rows removed: {removed_duplicates:,}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "features_final.csv"
features_final.to_csv(output_path, index=False)

if not output_path.exists():
    raise OSError(f"Output file was not created: {output_path}")

print(f"\nFile saved successfully: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024 * 1024):.2f} MB")
print("\nMissing values in final dataset:")
display(features_final.isna().sum().sort_values(ascending=False).head(10))


## Conclusion

This notebook prepared the validated clickstream and products data for modeling by merging the datasets, encoding categorical columns using appropriate strategies (ordinal for event types, frequency for high-cardinality features, one-hot for day of week), and normalizing numeric features to a common scale. We also performed exploratory data analysis to understand data distributions and patterns, saving key visualizations for the final report. The final processed dataset with all engineered features is now ready for the feature engineering stage to create user and product embeddings for the recommendation system.